# StoreListener: Bilingual Speech Recognition Pipeline

<a href="https://colab.research.google.com/github/MuhammadSarfrazAfzal/whisper-finetune/blob/main/storelistener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook implements the complete, robust parameter-efficient fine-tuning (PEFT / LoRA) and evaluation pipeline for **StoreListener** (Offline Automatic Speech Recognition for Retail).

### Objectives & Task Separation
* **Primary Training Objective (Phase 1 ASR)**:
  * English speech $\rightarrow$ English transcript (`task="transcribe"`, `<|en|><|transcribe|>`).
  * Spanish speech $\rightarrow$ Spanish transcript (`task="transcribe"`, `<|es|><|transcribe|>`).
* **Speech Translation (Phase 2 Task)**: Spanish speech $\rightarrow$ English text (`task="translate"`, `<|es|><|translate|>`) is strictly separated and tested as an independent zero-shot evaluation task.
* **Base Model**: `openai/whisper-base` (74M parameter multilingual checkpoint; `.en` checkpoints are excluded).
* **Dataset**: 600 verified clips (300 EN, 300 ES) from Google FLEURS; existing split assignments (488 train, 46 val, 66 test) are strictly preserved.
* **Compatibility & Stability**: Configured for modern Hugging Face Transformers and PEFT with automated smoke tests (finite loss and LoRA parameter gradient verification), Google Drive checkpoint persistence, and seamless Colab GPU execution.


## 1. Environment and Dependency Checks
Installs required libraries without version conflicts and performs hardware diagnostics (CUDA availability, GPU model, VRAM, and mixed-precision strategy).


In [ ]:
# Install required dependencies
%pip install -q "transformers>=4.48.0" "peft>=0.14.0" datasets accelerate evaluate jiwer soundfile

import sys
import os
import torch
import soundfile as sf
import datasets
import transformers
import peft
import accelerate
import evaluate

print("=" * 60)
print("ENVIRONMENT & DEPENDENCY DIAGNOSTICS")
print("Python:       ", sys.version.split()[0])
print("PyTorch:      ", torch.__version__)
print("CUDA build:   ", torch.version.cuda)
print("Transformers: ", transformers.__version__)
print("PEFT:         ", peft.__version__)
print("Datasets:     ", datasets.__version__)
print("Accelerate:   ", accelerate.__version__)
print("Evaluate:     ", evaluate.__version__)
print("SoundFile:    ", sf.__version__)
print("=" * 60)

CUDA_READY = torch.cuda.is_available()
USE_FP16 = False
USE_BF16 = False

if CUDA_READY:
    device_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    capability = torch.cuda.get_device_capability(0)

    USE_BF16 = torch.cuda.is_bf16_supported()
    USE_FP16 = not USE_BF16

    print("[+] Active GPU:         ", device_name)
    print(f"[+] Total VRAM:          {vram_gb:.2f} GB")
    print(f"[+] Compute Capability:  {capability[0]}.{capability[1]}")
    print("[+] Mixed Precision:    ", "BF16" if USE_BF16 else "FP16")
else:
    print("[!] WARNING: CUDA GPU not detected.")
    print("    In Google Colab: Runtime -> Change runtime type -> T4 GPU.")

print("=" * 60)


## 2. Google Drive Connection
Mounts Google Drive when executed in Google Colab to access dataset files and persist checkpoints.


In [ ]:
# Mount Google Drive when running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[+] Google Drive mounted successfully at /content/drive")
except ImportError:
    print("[i] google.colab not available: running in local environment.")


## 3. Existing Dataset Path Discovery & Output Directory
Discovers the existing dataset root in Google Drive or local workspace, and configures checkpoint and export output paths.


In [ ]:
from pathlib import Path

# Paths for Google Colab and local environments
COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive/whisper-finetune")
LOCAL_DEVELOPMENT_ROOT = Path(r"D:\Finetunning\whisper-finetune")
CWD_ROOT = Path(".").resolve()

CUSTOM_DATASET_ROOT = None  # Optional user override

if CUSTOM_DATASET_ROOT is not None and Path(CUSTOM_DATASET_ROOT).exists():
    DATASET_ROOT = Path(CUSTOM_DATASET_ROOT).resolve()
    print(f"[+] Using custom dataset root: {DATASET_ROOT}")
elif COLAB_DRIVE_ROOT.exists() and (COLAB_DRIVE_ROOT / "train.csv").exists():
    DATASET_ROOT = COLAB_DRIVE_ROOT
    print(f"[+] Using Google Drive Colab root: {DATASET_ROOT}")
elif LOCAL_DEVELOPMENT_ROOT.exists() and (LOCAL_DEVELOPMENT_ROOT / "train.csv").exists():
    DATASET_ROOT = LOCAL_DEVELOPMENT_ROOT
    print(f"[+] Using Local Development root: {DATASET_ROOT}")
elif (CWD_ROOT / "train.csv").exists():
    DATASET_ROOT = CWD_ROOT
    print(f"[+] Using Current Working Directory root: {DATASET_ROOT}")
else:
    raise FileNotFoundError(
        f"CRITICAL: train.csv not found in Colab Drive ({COLAB_DRIVE_ROOT}), "
        f"local path ({LOCAL_DEVELOPMENT_ROOT}), or cwd ({CWD_ROOT})."
    )

MODEL_ID = "openai/whisper-base"

# Output directory: Google Drive checkpoint folder if in Colab, otherwise local folder
if DATASET_ROOT == COLAB_DRIVE_ROOT:
    OUTPUT_DIR = COLAB_DRIVE_ROOT / "checkpoints" / "whisper-base-storelistener-lora"
else:
    OUTPUT_DIR = DATASET_ROOT / "checkpoints" / "whisper-base-storelistener-lora"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[+] Output directory ready: {OUTPUT_DIR}")


## 4. Dataset and Manifest Integrity Validation
Loads existing split manifests without modification, validates audio paths, verifies 16,000 Hz single-channel mono properties, and confirms UTF-8 transcripts.


In [ ]:
import pandas as pd
import numpy as np

# Load existing CSV manifests without modifying them
train_df = pd.read_csv(DATASET_ROOT / "train.csv")
val_df   = pd.read_csv(DATASET_ROOT / "validation.csv")
test_df  = pd.read_csv(DATASET_ROOT / "test.csv")
manifest_p = DATASET_ROOT / "dataset_manifest.csv"
manifest_df = pd.read_csv(manifest_p) if manifest_p.exists() else None

print("[+] Manifests Loaded:")
print(f"    Train Split:      {len(train_df):3d} rows (EN: {sum(train_df['language']=='en')}, ES: {sum(train_df['language']=='es')})")
print(f"    Validation Split: {len(val_df):3d} rows (EN: {sum(val_df['language']=='en')}, ES: {sum(val_df['language']=='es')})")
print(f"    Test Split:       {len(test_df):3d} rows (EN: {sum(test_df['language']=='en')}, ES: {sum(test_df['language']=='es')})")
print(f"    Total Rows:       {len(train_df) + len(val_df) + len(test_df)} (Expected: 600)")

assert len(train_df) == 488, f"Expected 488 train samples, found {len(train_df)}"
assert len(val_df) == 46, f"Expected 46 validation samples, found {len(val_df)}"
assert len(test_df) == 66, f"Expected 66 test samples, found {len(test_df)}"

# Split isolation verification
assert len(set(train_df["audio_path"]) & set(val_df["audio_path"])) == 0, "Leakage: train and val overlap!"
assert len(set(train_df["audio_path"]) & set(test_df["audio_path"])) == 0, "Leakage: train and test overlap!"
assert len(set(val_df["audio_path"]) & set(test_df["audio_path"])) == 0, "Leakage: val and test overlap!"
print("[+] Split isolation verified: 0 overlapping audio paths across splits.")

def resolve_audio_path(rel_path: str, dataset_root: Path) -> Path:
    """Robust audio path resolver supporting various folder structures."""
    p = Path(rel_path)
    candidates = [
        dataset_root / p,
        dataset_root / "datasets" / p,
        dataset_root / "datasets" / "audio" / p.parts[-2] / p.name if len(p.parts) >= 2 else None,
        dataset_root / "audio" / p.parts[-2] / p.name if len(p.parts) >= 2 else None,
        dataset_root / p.name,
    ]
    for c in candidates:
        if c is not None and c.exists():
            return c
    raise FileNotFoundError(f"Audio clip not found: {rel_path} under root {dataset_root}")

# Verify audio sample format and UTF-8 transcripts
en_sample = train_df[train_df["language"] == "en"].iloc[0]
es_sample = train_df[train_df["language"] == "es"].iloc[0]

en_path = resolve_audio_path(en_sample["audio_path"], DATASET_ROOT)
es_path = resolve_audio_path(es_sample["audio_path"], DATASET_ROOT)

en_wav, en_sr = sf.read(str(en_path), dtype="float32")
es_wav, es_sr = sf.read(str(es_path), dtype="float32")

assert en_sr == 16000 and en_wav.ndim == 1, "English audio must be 16 kHz Mono!"
assert es_sr == 16000 and es_wav.ndim == 1, "Spanish audio must be 16 kHz Mono!"

print("[+] Audio format verified (16000 Hz mono):")
print(f"    EN sample ({en_path.name}): duration = {len(en_wav)/en_sr:.2f}s")
print(f"    ES sample ({es_path.name}): duration = {len(es_wav)/es_sr:.2f}s")
print("[+] Sample transcripts:")
print(f"    EN: {en_sample['transcript'][:70]}...")
print(f"    ES: {es_sample['transcript'][:70]}...")


## 5. Train / Validation / Test Split Verification
Verifies data split sizes, language balance, and confirms zero empty transcripts across all partitions.


In [ ]:
# Split distribution and label integrity check
for split_name, split_df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    en_cnt = sum(split_df["language"] == "en")
    es_cnt = sum(split_df["language"] == "es")
    empty_transcripts = split_df["transcript"].isna().sum() + (split_df["transcript"].str.strip() == "").sum()
    print(f"[{split_name}] {len(split_df):3d} total | EN: {en_cnt:3d} | ES: {es_cnt:3d} | Empty transcripts: {empty_transcripts}")
    assert empty_transcripts == 0, f"Found empty transcripts in {split_name} split!"

print("\n[+] Verification complete: Splits are balanced and ready for preprocessing.")


## 6. Multilingual Whisper Processor and Base Model Setup
Loads the multilingual `openai/whisper-base` feature extractor, tokenizer, processor, and base model for bilingual ASR transcription.


In [ ]:
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration
)

TARGET_TASK = "transcribe"

print(f"[*] Loading feature extractor, tokenizer, and processor for: {MODEL_ID}")
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, task=TARGET_TASK)
processor = WhisperProcessor.from_pretrained(MODEL_ID, task=TARGET_TASK)

base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, low_cpu_mem_usage=True)

DECODER_START_TOKEN_ID = base_model.config.decoder_start_token_id
print(f"[+] Decoder start token ID: {DECODER_START_TOKEN_ID} ('{tokenizer.decode([DECODER_START_TOKEN_ID])}')")

# Unsuppress tokens and disable forced language in base model so both English and Spanish are supported
base_model.config.forced_decoder_ids = None
base_model.config.suppress_tokens = []

if hasattr(base_model, "generation_config") and base_model.generation_config is not None:
    base_model.generation_config.forced_decoder_ids = None
    base_model.generation_config.language = None
    base_model.generation_config.task = TARGET_TASK
    base_model.generation_config.suppress_tokens = []

en_prompts = tokenizer.get_decoder_prompt_ids(language="en", task=TARGET_TASK)
es_prompts = tokenizer.get_decoder_prompt_ids(language="es", task=TARGET_TASK)
print(f"[+] English prompt token IDs: {en_prompts}")
print(f"[+] Spanish prompt token IDs: {es_prompts}")


## 7. Audio Preprocessing Pipeline
Defines `StoreListenerASRDataset` to load waveforms, apply stereo-to-mono conversion, enforce the 30-second window, compute log-mel spectrogram features, and prepare sequence labels with prompt tokens.


In [ ]:
from torch.utils.data import Dataset

EXPECTED_SAMPLE_RATE = 16000
MAX_DURATION_SECONDS = 30.0
MAX_SAMPLES = int(EXPECTED_SAMPLE_RATE * MAX_DURATION_SECONDS)

class StoreListenerASRDataset(Dataset):
    def __init__(self, df, dataset_root, feature_extractor, tokenizer):
        self.df = df.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.feature_extractor = feature_extractor
        self.tokenizer = tokenizer

        self.prompt_cache = {
            "en": [tok for _, tok in tokenizer.get_decoder_prompt_ids(language="en", task="transcribe")],
            "es": [tok for _, tok in tokenizer.get_decoder_prompt_ids(language="es", task="transcribe")],
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        full_path = resolve_audio_path(row["audio_path"], self.dataset_root)

        waveform, sr = sf.read(str(full_path), dtype="float32")
        if sr != EXPECTED_SAMPLE_RATE:
            raise ValueError(f"{full_path.name}: {sr} Hz found, expected {EXPECTED_SAMPLE_RATE} Hz")

        if waveform.ndim == 2:
            waveform = waveform.mean(axis=1)

        if waveform.ndim != 1 or len(waveform) == 0:
            raise ValueError(f"Invalid audio waveform: {full_path}")

        # Truncate clips longer than 30 seconds
        waveform = waveform[:MAX_SAMPLES]

        # Extract log-mel spectrogram features: shape (80, 3000)
        input_features = self.feature_extractor(
            waveform,
            sampling_rate=EXPECTED_SAMPLE_RATE
        ).input_features[0]

        lang = str(row["language"]).strip().lower()
        if lang not in self.prompt_cache:
            raise ValueError(f"Unsupported language '{lang}' in dataset row {idx}")

        transcript = str(row["transcript"]).strip()
        if not transcript:
            raise ValueError(f"Empty transcript in dataset row {idx}")

        text_token_ids = self.tokenizer(transcript, add_special_tokens=False).input_ids

        # Label sequence: [language_token, task_token, notimestamps_token] + text_tokens + [eos_token]
        labels = self.prompt_cache[lang] + text_token_ids + [self.tokenizer.eos_token_id]

        return {
            "input_features": input_features,
            "labels": labels,
            "language": lang,
        }

train_dataset = StoreListenerASRDataset(train_df, DATASET_ROOT, feature_extractor, tokenizer)
val_dataset   = StoreListenerASRDataset(val_df, DATASET_ROOT, feature_extractor, tokenizer)
test_dataset  = StoreListenerASRDataset(test_df, DATASET_ROOT, feature_extractor, tokenizer)

print(f"[+] Datasets ready: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
sample_item = train_dataset[0]
print(f"    Sample features shape: {sample_item['input_features'].shape}")
print(f"    Sample labels count:   {len(sample_item['labels'])}")
print(f"    Sample language:       {sample_item['language']}")


## 8. Data Collator Initialization
Batches audio features and label sequences. Masks padding tokens with `-100` so they are ignored by `CrossEntropyLoss`. Returns strictly `input_features` and `labels` to prevent keyword argument collisions.


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Mask padding with -100 so CrossEntropyLoss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        # Strip decoder start token if present at index 0 across all items in batch
        if labels.shape[1] > 0 and (labels[:, 0] == self.decoder_start_token_id).all().item():
            labels = labels[:, 1:]

        # Return strictly the tensors required by the model to prevent argument conflicts
        return {
            "input_features": batch["input_features"],
            "labels": labels,
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=DECODER_START_TOKEN_ID
)

# Sanity check with a 2-sample batch
test_batch = data_collator([train_dataset[0], train_dataset[1]])
print("[+] Data Collator Sanity Check:")
print("    input_features shape:", tuple(test_batch["input_features"].shape))
print("    labels shape:        ", tuple(test_batch["labels"].shape))
print("    Masked tokens (-100):", int((test_batch["labels"] == -100).sum()))

assert test_batch["input_features"].shape == (2, 80, 3000), "Invalid feature batch shape!"
assert "input_ids" not in test_batch, "Collator must not return input_ids for Whisper encoder!"
assert torch.isfinite(test_batch["input_features"]).all(), "Non-finite input features found!"
print("[+] Data collator verification passed.")


## 9. Parameter-Efficient Fine-Tuning (PEFT / LoRA) Configuration
Applies Low-Rank Adaptation (LoRA) to `q_proj` and `v_proj` attention projection layers of `openai/whisper-base` ($r=16, \alpha=32$).

> [!NOTE]
> **Resolution of `WhisperDecoder ... multiple values for keyword argument 'input_ids'`**:
> Whisper is an audio Seq2Seq model whose encoder takes `input_features`, not text `input_ids`. Setting `task_type=TaskType.SEQ_2_SEQ_LM` causes PEFT to wrap Whisper in `PeftModelForSeq2SeqLM`, which assumes text Seq2Seq models and injects `input_ids=None`. In modern Transformers, this unused keyword argument is passed down via `kwargs` into `WhisperDecoder.forward(input_ids=decoder_input_ids, **kwargs)`, colliding with the explicit `input_ids` parameter.
> Omitting `task_type` instructs PEFT to instantiate generic `PeftModel`, cleanly preserving Whisper's native `input_features` forward signature without monkey patching.


In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

# Configure LoRA for Whisper's attention projections
# Note: task_type is intentionally omitted to avoid text-LM argument injection
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

def build_lora_whisper_model(model: WhisperForConditionalGeneration, peft_config: LoraConfig) -> PeftModel:
    """
    Applies gradient checkpointing and attaches LoRA adapters to Whisper without
    injecting text-centric input_ids into the audio encoder.
    """
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    if hasattr(model, "peft_config"):
        print("[i] Model already wrapped with PEFT. Re-wrapping with provided config...")

    lora_model = get_peft_model(model, peft_config)
    lora_model.print_trainable_parameters()
    return lora_model

lora_model = build_lora_whisper_model(base_model, lora_config)

if CUDA_READY:
    lora_model = lora_model.to("cuda")
    print(f"[+] Model moved to active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[i] Model initialized on CPU.")


## 10. Forward-Pass and Backward-Pass Smoke Tests
Executes single-batch forward and backward passes before training to verify that the forward pass computes a finite loss and that backpropagation produces valid gradients on the intended LoRA parameters.


In [ ]:
# Real single-batch forward and backward pass smoke test
device = "cuda" if CUDA_READY else "cpu"
lora_model.train()

smoke_batch = data_collator([train_dataset[0], train_dataset[1]])
smoke_batch = {k: v.to(device) for k, v in smoke_batch.items()}

lora_model.zero_grad(set_to_none=True)

with torch.enable_grad():
    outputs = lora_model(
        input_features=smoke_batch["input_features"],
        labels=smoke_batch["labels"]
    )
    loss = outputs.loss

print(f"[+] Forward-pass smoke test completed.")
print(f"    Loss value:  {loss.item():.4f}")
print(f"    Loss finite: {torch.isfinite(loss).item()}")

assert loss is not None and torch.isfinite(loss), "CRITICAL: Model produced non-finite or null loss!"

# Test backward pass to verify adapter gradients
loss.backward()

grad_params = [
    (name, param) for name, param in lora_model.named_parameters()
    if param.requires_grad and param.grad is not None
]

print(f"[+] Backward-pass smoke test completed.")
print(f"    Trainable parameters receiving gradients: {len(grad_params)}")

assert len(grad_params) > 0, "CRITICAL: No parameters received gradients during backward pass!"

# Verify that gradients belong to LoRA modules
lora_grads = [name for name, _ in grad_params if "lora_" in name]
assert len(lora_grads) > 0, "CRITICAL: LoRA parameters did not receive gradients!"
print(f"    LoRA gradient tensors verified: {len(lora_grads)}")

lora_model.zero_grad(set_to_none=True)
print("[+] Smoke tests PASSED: Model is ready for training.")


## 11. Evaluation Metrics and Seq2SeqTrainer Setup
Defines Word Error Rate (WER) and Character Error Rate (CER) metric computation (normalized and verbatim), broken down by language, and configures `Seq2SeqTrainer`.


In [ ]:
import inspect
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")
normalizer = BasicTextNormalizer()

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # Identify sample language from first valid prompt token before replacing -100
    languages = []
    for l_seq in label_ids:
        valid_toks = [tok for tok in l_seq if tok != -100]
        if len(valid_toks) > 0:
            tok_str = tokenizer.decode([valid_toks[0]], skip_special_tokens=False)
            languages.append("es" if "es" in tok_str else "en")
        else:
            languages.append("en")

    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
    pred_ids  = np.where(pred_ids  != -100, pred_ids,  tokenizer.pad_token_id)

    pred_str  = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    overall_wer_verb = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    overall_cer_verb = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    pred_str_norm  = [normalizer(p) for p in pred_str]
    label_str_norm = [normalizer(l) for l in label_str]

    valid_pairs = [(p, l) for p, l in zip(pred_str_norm, label_str_norm) if len(l.strip()) > 0]
    if valid_pairs:
        p_norm, l_norm = zip(*valid_pairs)
        overall_wer_norm = 100 * wer_metric.compute(predictions=list(p_norm), references=list(l_norm))
        overall_cer_norm = 100 * cer_metric.compute(predictions=list(p_norm), references=list(l_norm))
    else:
        overall_wer_norm, overall_cer_norm = 0.0, 0.0

    # Language-specific breakdowns
    en_preds_norm = [p for p, lang, l in zip(pred_str_norm, languages, label_str_norm) if lang == "en" and len(l.strip()) > 0]
    en_refs_norm  = [l for l, lang in zip(label_str_norm, languages) if lang == "en" and len(l.strip()) > 0]
    es_preds_norm = [p for p, lang, l in zip(pred_str_norm, languages, label_str_norm) if lang == "es" and len(l.strip()) > 0]
    es_refs_norm  = [l for l, lang in zip(label_str_norm, languages) if lang == "es" and len(l.strip()) > 0]

    en_wer = 100 * wer_metric.compute(predictions=en_preds_norm, references=en_refs_norm) if en_refs_norm else 0.0
    en_cer = 100 * cer_metric.compute(predictions=en_preds_norm, references=en_refs_norm) if en_refs_norm else 0.0
    es_wer = 100 * wer_metric.compute(predictions=es_preds_norm, references=es_refs_norm) if es_refs_norm else 0.0
    es_cer = 100 * cer_metric.compute(predictions=es_preds_norm, references=es_refs_norm) if es_refs_norm else 0.0

    return {
        "wer_normalized": round(overall_wer_norm, 2),
        "cer_normalized": round(overall_cer_norm, 2),
        "en_wer":         round(en_wer, 2),
        "en_cer":         round(en_cer, 2),
        "es_wer":         round(es_wer, 2),
        "es_cer":         round(es_cer, 2),
        "wer_verbatim":   round(overall_wer_verb, 2),
        "cer_verbatim":   round(overall_cer_verb, 2),
    }

# =====================================================================
# Configurable Training Hyperparameters
# =====================================================================
NUM_TRAIN_EPOCHS = 3           # Training epochs
MAX_STEPS = -1                 # Set > 0 (e.g. 50) for smoke tests, or -1 to use NUM_TRAIN_EPOCHS
TRAIN_BATCH_SIZE = 8           # Conservative batch size for Whisper-base on T4 GPU
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2 # Effective batch size: 16
LEARNING_RATE = 1e-4
WARMUP_STEPS = 50
LOGGING_STEPS = 10
SAVE_STEPS = 100
EVAL_STEPS = 100
RESUME_FROM_CHECKPOINT = True  # Automatically resume from latest compatible checkpoint in OUTPUT_DIR if present

eval_strat_key = "eval_strategy" if hasattr(Seq2SeqTrainingArguments, "eval_strategy") else "evaluation_strategy"
training_args_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": WARMUP_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "gradient_checkpointing": True,
    "fp16": USE_FP16,
    "bf16": USE_BF16,
    eval_strat_key: "steps",
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "save_total_limit": 2,
    "logging_steps": LOGGING_STEPS,
    "report_to": ["none"],
    "predict_with_generate": True,
    "generation_max_length": 225,
    "remove_unused_columns": False,
    "dataloader_num_workers": 2 if sys.platform != "win32" else 0,
    "load_best_model_at_end": False,
}
if MAX_STEPS > 0:
    training_args_kwargs["max_steps"] = MAX_STEPS

training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

def build_trainer(model, train_data, eval_data, collator, metrics_callback, args, proc):
    """Initializes Seq2SeqTrainer with cross-version processor/tokenizer compatibility."""
    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_data,
        "eval_dataset": eval_data,
        "data_collator": collator,
        "compute_metrics": metrics_callback,
    }
    # Transformers 4.48+ / 5.x uses processing_class; earlier versions use tokenizer
    sig = inspect.signature(Seq2SeqTrainer.__init__)
    if "processing_class" in sig.parameters:
        trainer_kwargs["processing_class"] = proc
    else:
        trainer_kwargs["tokenizer"] = proc.feature_extractor

    return Seq2SeqTrainer(**trainer_kwargs)

trainer = build_trainer(
    model=lora_model,
    train_data=train_dataset,
    eval_data=val_dataset,
    collator=data_collator,
    metrics_callback=compute_metrics,
    args=training_args,
    proc=processor,
)

print(f"[+] Seq2SeqTrainer initialized successfully. Checkpoints will save to: {OUTPUT_DIR}")


## 12. Model Fine-Tuning Execution
Starts Whisper LoRA fine-tuning. Automatically resumes from the latest checkpoint if one exists in `OUTPUT_DIR`.


In [ ]:
from transformers.trainer_utils import get_last_checkpoint

# =====================================================================
# Training Execution Control
# =====================================================================
EXECUTE_TRAINING = True  # Set to True to train; False to skip training and keep pipeline read-only

print("=" * 70)
print("STORELISTENER TRAINING GATE")
print(f"  EXECUTE_TRAINING:       {EXECUTE_TRAINING}")
print(f"  CUDA_READY:             {CUDA_READY}")
print(f"  DATASET_ROOT:           {DATASET_ROOT}")
print(f"  OUTPUT_DIR:             {OUTPUT_DIR}")
print(f"  RESUME_FROM_CHECKPOINT: {RESUME_FROM_CHECKPOINT}")
print("=" * 70)

if not EXECUTE_TRAINING:
    print("\n[i] Training execution is DISABLED (EXECUTE_TRAINING = False).")
    print("    Environment, data collation, forward/backward smoke tests, and trainer passed.")
    print("    Set EXECUTE_TRAINING = True when ready to train.")
elif not CUDA_READY:
    raise RuntimeError(
        "CRITICAL: CUDA GPU not detected! Training Whisper cannot run on CPU. "
        "In Google Colab: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU, then re-run."
    )
else:
    checkpoint_to_resume = None
    if RESUME_FROM_CHECKPOINT:
        last_ckpt = get_last_checkpoint(str(OUTPUT_DIR))
        if last_ckpt is not None:
            checkpoint_to_resume = last_ckpt
            print(f"[+] Found existing checkpoint. Resuming from: {checkpoint_to_resume}")
        else:
            print("[+] No existing checkpoint found in OUTPUT_DIR. Starting fresh training run.")

    print("\n[*] Starting Whisper LoRA training...")
    train_result = trainer.train(resume_from_checkpoint=checkpoint_to_resume)
    print("\n[+] Training completed successfully!")
    print(f"    Global steps:    {train_result.global_step}")
    print(f"    Training loss:   {train_result.training_loss:.4f}")


## 13. Checkpoint and LoRA Adapter Export
Exports the final LoRA adapter weights, adapter configuration, and processor to `OUTPUT_DIR`. Verifies that the saved files exist and can be reloaded.


In [ ]:
def export_checkpoints(model, proc, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(output_dir))
    proc.save_pretrained(str(output_dir))
    print(f"[+] LoRA adapter and processor saved to: {output_dir}")

if EXECUTE_TRAINING and CUDA_READY:
    export_checkpoints(lora_model, processor, OUTPUT_DIR)

    # Verification: Verify files exist and test reload
    adapter_files = list(OUTPUT_DIR.glob("*"))
    print(f"[+] Output directory contents ({len(adapter_files)} items):")
    for f in adapter_files:
        print(f"    - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

    print("\n[*] Verifying that exported adapter can be reloaded...")
    reload_base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, low_cpu_mem_usage=True)
    reloaded_model = PeftModel.from_pretrained(reload_base, str(OUTPUT_DIR))
    reloaded_model.eval()
    print("[+] Verification SUCCESS: Adapter loaded cleanly from output directory!")
else:
    print("[i] Training was skipped or CUDA unavailable; export skipped.")


## 14. Validation Set Evaluation
Evaluates the model on `validation.csv` (46 samples) and reports Word Error Rate and Character Error Rate broken down by English and Spanish.


In [ ]:
print("[*] Running evaluation on Validation Set (46 samples)...")
val_results = trainer.evaluate(eval_dataset=val_dataset, metric_key_prefix="val")

print("\n" + "=" * 55)
print("VALIDATION METRICS SUMMARY")
print("=" * 55)
for k, v in val_results.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")
    else:
        print(f"  {k:25s}: {v}")
print("=" * 55)


## 15. Held-Out Test Set Evaluation
Evaluates the model on `test.csv` (66 samples) strictly as a held-out evaluation set, verifying unbiased generalizability across English and Spanish.


In [ ]:
print("[*] Running evaluation on Held-Out Test Set (66 samples)...")
test_results = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")

print("\n" + "=" * 55)
print("HELD-OUT TEST SET METRICS SUMMARY")
print("=" * 55)
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")
    else:
        print(f"  {k:25s}: {v}")
print("=" * 55)


## 16. English Transcription and Spanish-Audio Translation Checks
Demonstrates qualitative model inference on test samples:
- English audio $\rightarrow$ English transcription (`task="transcribe"`)
- Spanish audio $\rightarrow$ Spanish transcription (`task="transcribe"`)
- Spanish audio $\rightarrow$ English translation (`task="translate"`)

> [!NOTE]
> **Task Boundary & Dataset Limitation**:
> `test.csv` contains Spanish transcripts for Spanish audio, not reference English translations. While Whisper's zero-shot translation capability (`task="translate"`) is demonstrated qualitatively below, quantitative translation metrics (e.g. translation WER/BLEU) cannot be calculated without reference translations.


In [ ]:
def run_inference_sample(model, proc, row, task="transcribe", target_device=None):
    if target_device is None:
        target_device = "cuda" if torch.cuda.is_available() else "cpu"

    audio_p = resolve_audio_path(row["audio_path"], DATASET_ROOT)
    wav, sr = sf.read(str(audio_p), dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = wav[:MAX_SAMPLES]

    inputs = proc.feature_extractor(
        wav, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(target_device)

    src_lang = str(row["language"]).strip().lower()
    lang_full = "english" if src_lang == "en" else "spanish"

    model.eval()
    with torch.no_grad():
        gen_tokens = model.generate(
            input_features=inputs,
            language=lang_full,
            task=task,
            max_new_tokens=128
        )

    return proc.tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)[0].strip()

# Run sample demonstrations
print("=" * 70)
print("QUALITATIVE INFERENCE DEMONSTRATIONS")
print("=" * 70)

# 1. English Transcription
en_test_row = test_df[test_df["language"] == "en"].iloc[0]
en_pred = run_inference_sample(lora_model, processor, en_test_row, task="transcribe", target_device=device)
print(f"\n[EN Transcription]")
print(f"  Audio:     {en_test_row['audio_path']}")
print(f"  Reference: {en_test_row['transcript']}")
print(f"  Predicted: {en_pred}")

# 2. Spanish Transcription
es_test_row = test_df[test_df["language"] == "es"].iloc[0]
es_pred = run_inference_sample(lora_model, processor, es_test_row, task="transcribe", target_device=device)
print(f"\n[ES Transcription]")
print(f"  Audio:     {es_test_row['audio_path']}")
print(f"  Reference: {es_test_row['transcript']}")
print(f"  Predicted: {es_pred}")

# 3. Spanish -> English Translation (Zero-shot Phase 2 task)
es_trans_pred = run_inference_sample(lora_model, processor, es_test_row, task="translate", target_device=device)
print(f"\n[ES -> EN Translation (Zero-Shot Phase 2)]")
print(f"  Audio:           {es_test_row['audio_path']}")
print(f"  Spanish Source:  {es_test_row['transcript']}")
print(f"  Translated Text: {es_trans_pred}")
print("\n[i] Note: Quantitative translation WER is not evaluated because dataset contains only Spanish reference text.")
print("=" * 70)


## 17. Final Training Summary and Artifact Verification
Summarizes execution status, split sizes, LoRA hyperparameter configuration, saved checkpoint artifacts, and evaluation metrics.


In [ ]:
print("=" * 75)
print("STORELISTENER PIPELINE EXECUTION SUMMARY")
print("=" * 75)
print(f"Base Model:             {MODEL_ID}")
print(f"Dataset Root:           {DATASET_ROOT}")
print(f"Split Verification:     Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"LoRA Target Modules:    q_proj, v_proj (r=16, alpha=32)")
print(f"Output Directory:       {OUTPUT_DIR}")
print(f"Training Executed:      {EXECUTE_TRAINING}")
print(f"CUDA Available:         {CUDA_READY}")

if OUTPUT_DIR.exists():
    exported = [f.name for f in OUTPUT_DIR.glob("*") if f.is_file()]
    print(f"Exported Artifacts:     {exported}")

print("=" * 75)
print("[+] StoreListener Whisper LoRA pipeline execution complete.")
